<a href="https://colab.research.google.com/github/mennamoamenn1/architech-core/blob/main/01_foundation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install ortools shapely matplotlib pydantic pyyaml -q

In [2]:
from pydantic import BaseModel, Field, field_validator, model_validator
from enum import Enum
from typing import Optional


class BuildingType(str, Enum):
    APARTMENT = "apartment"
    VILLA = "villa"
    RESIDENTIAL_BUILDING = "residential_building"
    COMPOUND = "compound"
    MALL = "mall"
    OFFICE = "office"
    CLINIC = "clinic"


class RoomType(str, Enum):
    LIVING = "living"
    RECEPTION = "reception"
    BEDROOM = "bedroom"
    MASTER_BEDROOM = "master_bedroom"
    KITCHEN = "kitchen"
    BATHROOM = "bathroom"
    DINING = "dining"
    CORRIDOR = "corridor"
    BALCONY = "balcony"
    SHOP = "shop"


class GreenElement(str, Enum):
    POOL = "pool"
    FOUNTAIN = "fountain"
    GARDEN = "garden"
    TREES = "trees"
    COURTYARD = "courtyard"


class ConstraintKind(str, Enum):
    ADJACENT = "adjacent"            # a must touch b
    NOT_ADJACENT = "not_adjacent"    # a must not touch b
    NEAR_ENTRANCE = "near_entrance"  # only needs a
    FACES_SOUTH = "faces_south"      # only needs a


class RoomRequest(BaseModel):
    type: RoomType
    count: int = Field(1, ge=1, le=50)
    min_area: Optional[float] = Field(None, gt=0)  # m²; None = use the rules file


class Constraint(BaseModel):
    kind: ConstraintKind
    a: RoomType
    b: Optional[RoomType] = None

    @model_validator(mode="after")
    def check_b(self):
        needs_b = self.kind in (ConstraintKind.ADJACENT, ConstraintKind.NOT_ADJACENT)
        if needs_b and self.b is None:
            raise ValueError(f"constraint '{self.kind.value}' needs a second room 'b'")
        if not needs_b and self.b is not None:
            raise ValueError(f"constraint '{self.kind.value}' must not have a room 'b'")
        return self


class PlotSpec(BaseModel):
    width: float = Field(..., gt=0, le=2000)   # meters
    depth: float = Field(..., gt=0, le=2000)
    zone: str = Field("default", max_length=50)  # city / planning zone for code rules


class ProjectSpec(BaseModel):
    building_type: BuildingType
    plot: PlotSpec
    floors: int = Field(1, ge=1, le=40)
    rooms: list[RoomRequest]
    constraints: list[Constraint] = []
    green_elements: list[GreenElement] = []
    num_alternatives: int = Field(3, ge=1, le=10)

    @field_validator("rooms")
    @classmethod
    def need_rooms(cls, v):
        if not v:
            raise ValueError("At least one room is required")
        return v

    @model_validator(mode="after")
    def constraints_refer_to_requested_rooms(self):
        requested = {r.type for r in self.rooms}
        for c in self.constraints:
            for room in (c.a, c.b):
                if room is not None and room not in requested:
                    raise ValueError(
                        f"constraint '{c.kind.value}' mentions '{room.value}' "
                        f"but no such room was requested"
                    )
        return self


print("Model loaded OK")

Model loaded OK


In [3]:
spec = ProjectSpec(
    building_type="villa",
    plot=PlotSpec(width=20, depth=25, zone="new_cairo"),
    floors=2,
    rooms=[
        RoomRequest(type="living"),
        RoomRequest(type="dining"),
        RoomRequest(type="bedroom", count=3),
        RoomRequest(type="bathroom", count=3),
        RoomRequest(type="kitchen"),
    ],
    constraints=[Constraint(kind="adjacent", a="kitchen", b="dining")],
    green_elements=["pool", "garden"],
)

# 1) Pretty JSON
print(spec.model_dump_json(indent=2))

# 2) Access fields like normal Python objects
print("Plot area:", spec.plot.width * spec.plot.depth, "m²")
print("Total rooms:", sum(r.count for r in spec.rooms))
print("Type of building_type:", type(spec.building_type))

# 3) Round trip: JSON text -> object (this is what your frontend will send later)
json_text = spec.model_dump_json()
spec2 = ProjectSpec.model_validate_json(json_text)
print("Round trip equal:", spec == spec2)

{
  "building_type": "villa",
  "plot": {
    "width": 20.0,
    "depth": 25.0,
    "zone": "new_cairo"
  },
  "floors": 2,
  "rooms": [
    {
      "type": "living",
      "count": 1,
      "min_area": null
    },
    {
      "type": "dining",
      "count": 1,
      "min_area": null
    },
    {
      "type": "bedroom",
      "count": 3,
      "min_area": null
    },
    {
      "type": "bathroom",
      "count": 3,
      "min_area": null
    },
    {
      "type": "kitchen",
      "count": 1,
      "min_area": null
    }
  ],
  "constraints": [
    {
      "kind": "adjacent",
      "a": "kitchen",
      "b": "dining"
    }
  ],
  "green_elements": [
    "pool",
    "garden"
  ],
  "num_alternatives": 3
}
Plot area: 500.0 m²
Total rooms: 9
Type of building_type: <enum 'BuildingType'>
Round trip equal: True


In [4]:
from pydantic import ValidationError

def expect_failure(name, **kwargs):
    try:
        ProjectSpec(**kwargs)
        print(f"❌ {name}: was ACCEPTED (it should have been rejected)")
    except ValidationError as e:
        msgs = [f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors()]
        print(f"✅ {name}: rejected -> {msgs}")

good_plot = {"width": 20, "depth": 25}
good_rooms = [{"type": "living"}, {"type": "kitchen"}]

expect_failure("negative width",
    building_type="villa", plot={"width": -5, "depth": 25}, rooms=good_rooms)

expect_failure("empty rooms",
    building_type="villa", plot=good_plot, rooms=[])

expect_failure("unknown building type",
    building_type="castle", plot=good_plot, rooms=good_rooms)

expect_failure("too many floors",
    building_type="villa", plot=good_plot, floors=500, rooms=good_rooms)

expect_failure("constraint on missing room",
    building_type="villa", plot=good_plot, rooms=good_rooms,
    constraints=[{"kind": "adjacent", "a": "kitchen", "b": "dining"}])

expect_failure("adjacent without b",
    building_type="villa", plot=good_plot, rooms=good_rooms,
    constraints=[{"kind": "adjacent", "a": "kitchen"}])

expect_failure("typo in constraint kind",
    building_type="villa", plot=good_plot, rooms=good_rooms,
    constraints=[{"kind": "adjecent", "a": "kitchen", "b": "living"}])

✅ negative width: rejected -> ['plot.width: Input should be greater than 0']
✅ empty rooms: rejected -> ['rooms: Value error, At least one room is required']
✅ unknown building type: rejected -> ["building_type: Input should be 'apartment', 'villa', 'residential_building', 'compound', 'mall', 'office' or 'clinic'"]
✅ too many floors: rejected -> ['floors: Input should be less than or equal to 40']
✅ constraint on missing room: rejected -> [": Value error, constraint 'adjacent' mentions 'dining' but no such room was requested"]
✅ adjacent without b: rejected -> ["constraints.0: Value error, constraint 'adjacent' needs a second room 'b'"]
✅ typo in constraint kind: rejected -> ["constraints.0.kind: Input should be 'adjacent', 'not_adjacent', 'near_entrance' or 'faces_south'"]


In [5]:
import yaml

rules_yaml = """
meta:
  version: 0.1
  status: DRAFT - values must be verified against official sources
rooms:
  bedroom:
    min_area: null      # TODO verify
    min_width: null     # TODO verify
    source: null
  kitchen:
    min_area: null
    min_width: null
    source: null
site:
  setbacks: {front: null, side: null, rear: null}
  max_coverage_ratio: null
  min_green_ratio: null
"""

RULES = yaml.safe_load(rules_yaml)


def get_rule(path: str):
    """get_rule('rooms.bedroom.min_area') -> value, or raises if unverified."""
    node = RULES
    for key in path.split("."):
        if not isinstance(node, dict) or key not in node:
            raise KeyError(f"Rule not defined: {path}")
        node = node[key]
    if node is None:
        raise ValueError(f"Rule '{path}' is not verified yet (null). Fill it from an official source.")
    return node


# Tests
for p in ["rooms.bedroom.min_area", "site.setbacks.front", "rooms.garage.min_area"]:
    try:
        print(p, "=", get_rule(p))
    except Exception as e:
        print(f"✅ {type(e).__name__}: {e}")

✅ ValueError: Rule 'rooms.bedroom.min_area' is not verified yet (null). Fill it from an official source.
✅ ValueError: Rule 'site.setbacks.front' is not verified yet (null). Fill it from an official source.
✅ KeyError: 'Rule not defined: rooms.garage.min_area'
